# EcoHome Energy Advisor — Database Setup

This notebook initialises the SQLite database and populates it with realistic
sample data for energy usage, solar generation, device configurations, and
electricity pricing.

**Run this notebook first before any other notebook.**

In [ ]:
import os, sys, random
from datetime import datetime, timedelta

# Ensure project root is on sys.path
sys.path.insert(0, os.path.abspath('..'))

from sqlalchemy.orm import Session
from models.energy import (
    Base, EnergyUsage, SolarGeneration, DeviceUsage, ElectricityPrice,
    init_db, get_engine
)

DB_PATH = 'data/energy_data.db'
os.makedirs('data', exist_ok=True)

engine = init_db(DB_PATH)
print('✅ Database created at:', DB_PATH)

## 1. Seed Device Configurations

In [ ]:
DEVICES = [
    dict(device_name='Central AC', device_category='HVAC', rated_power_kw=3.5,
         preferred_start_hour=10, preferred_end_hour=15, is_schedulable=1,
         notes='Pre-cool during solar peak hours'),
    dict(device_name='Heat Pump', device_category='HVAC', rated_power_kw=2.8,
         preferred_start_hour=10, preferred_end_hour=14, is_schedulable=1,
         notes='Efficient at mild temperatures; avoid peak hours'),
    dict(device_name='EV Charger (Level 2)', device_category='EV', rated_power_kw=7.2,
         preferred_start_hour=22, preferred_end_hour=6, is_schedulable=1,
         notes='Prefer off-peak or solar peak charging'),
    dict(device_name='Pool Pump', device_category='Appliance', rated_power_kw=1.5,
         preferred_start_hour=10, preferred_end_hour=16, is_schedulable=1,
         notes='Run 6-8 hours; solar hours ideal'),
    dict(device_name='Dishwasher', device_category='Appliance', rated_power_kw=1.2,
         preferred_start_hour=21, preferred_end_hour=23, is_schedulable=1,
         notes='Delay-start to off-peak or after 9 PM'),
    dict(device_name='Washing Machine', device_category='Appliance', rated_power_kw=0.5,
         preferred_start_hour=10, preferred_end_hour=14, is_schedulable=1,
         notes='Cold water wash; midday solar preferred'),
    dict(device_name='Dryer', device_category='Appliance', rated_power_kw=5.0,
         preferred_start_hour=21, preferred_end_hour=23, is_schedulable=1,
         notes='High load; run at off-peak or super off-peak'),
    dict(device_name='Water Heater (Heat Pump)', device_category='Appliance', rated_power_kw=0.5,
         preferred_start_hour=10, preferred_end_hour=15, is_schedulable=1,
         notes='Run during solar peak for free hot water'),
    dict(device_name='Refrigerator', device_category='Appliance', rated_power_kw=0.15,
         preferred_start_hour=None, preferred_end_hour=None, is_schedulable=0,
         notes='Always-on; optimise with door seal checks'),
    dict(device_name='Lighting (LED)', device_category='Lighting', rated_power_kw=0.3,
         preferred_start_hour=None, preferred_end_hour=None, is_schedulable=0,
         notes='Smart dimming and occupancy sensors recommended'),
]

with Session(engine) as session:
    for d in DEVICES:
        if not session.query(DeviceUsage).filter_by(device_name=d['device_name']).first():
            session.add(DeviceUsage(**d))
    session.commit()

print(f'✅ Seeded {len(DEVICES)} device configurations.')

## 2. Generate 30 Days of Electricity Pricing Data

In [ ]:
def tou_price(hour, is_weekend):
    if is_weekend:
        if 9 <= hour <= 21:
            return 'off-peak', round(random.uniform(0.12, 0.16), 4)
        return 'super-off-peak', round(random.uniform(0.07, 0.10), 4)
    else:
        if 7 <= hour <= 9 or 17 <= hour <= 21:
            return 'peak', round(random.uniform(0.28, 0.38), 4)
        elif 10 <= hour <= 16:
            return 'off-peak', round(random.uniform(0.14, 0.18), 4)
        return 'super-off-peak', round(random.uniform(0.07, 0.11), 4)

base_date = datetime.now() - timedelta(days=30)
price_records = []

for day_offset in range(30):
    day = base_date + timedelta(days=day_offset)
    is_weekend = day.weekday() >= 5
    for hour in range(24):
        tier, price = tou_price(hour, is_weekend)
        price_records.append(ElectricityPrice(
            timestamp=day.replace(hour=hour, minute=0, second=0),
            hour_of_day=hour,
            price_per_kwh=price,
            tier=tier,
            day_type='weekend' if is_weekend else 'weekday'
        ))

with Session(engine) as session:
    session.add_all(price_records)
    session.commit()

print(f'✅ Inserted {len(price_records)} electricity price records (30 days × 24 hours).')

## 3. Generate 30 Days of Solar Generation Data

In [ ]:
PANEL_CAPACITY_KW = 10.0
CONDITIONS = ['Sunny', 'Partly Cloudy', 'Cloudy', 'Light Rain', 'Clear']
WEIGHTS    = [0.35, 0.30, 0.15, 0.10, 0.10]

def cloud_pct(condition):
    return {'Sunny': 5, 'Partly Cloudy': 40, 'Cloudy': 80, 'Light Rain': 90, 'Clear': 2}[condition]

solar_records = []
base_date = datetime.now() - timedelta(days=30)

for day_offset in range(30):
    day = base_date + timedelta(days=day_offset)
    condition = random.choices(CONDITIONS, weights=WEIGHTS)[0]
    cp = cloud_pct(condition)
    temp = round(random.uniform(15, 38), 1)

    for hour in range(6, 20):  # daylight hours only
        solar_factor = max(0, 1 - abs(hour - 13) / 7)  # peak at 1 PM
        irradiance = round(solar_factor * (100 - cp) / 100 * random.uniform(800, 1000), 1)
        generation = round(irradiance / 1000 * PANEL_CAPACITY_KW * (1 - (temp - 25) * 0.004), 3)
        generation = max(0.0, generation)
        efficiency = round(generation / (PANEL_CAPACITY_KW * solar_factor + 0.001) * 100, 1) if solar_factor > 0.05 else 0.0

        solar_records.append(SolarGeneration(
            timestamp=day.replace(hour=hour, minute=0, second=0),
            generation_kwh=generation,
            panel_capacity_kw=PANEL_CAPACITY_KW,
            efficiency_pct=min(efficiency, 100.0),
            temperature_c=temp,
            cloud_cover_pct=float(cp),
            irradiance_wm2=irradiance,
        ))

with Session(engine) as session:
    session.add_all(solar_records)
    session.commit()

print(f'✅ Inserted {len(solar_records)} solar generation records.')

## 4. Generate 30 Days of Energy Usage Data

In [ ]:
USAGE_PATTERNS = [
    # (device_name, category, power_kw, daily_hours, typical_start_hour, cost_rate)
    ('Central AC', 'HVAC', 3.5, 6, 13, 0.16),
    ('Heat Pump', 'HVAC', 2.8, 4, 7, 0.14),
    ('EV Charger (Level 2)', 'EV', 7.2, 4, 22, 0.09),
    ('Pool Pump', 'Appliance', 1.5, 7, 10, 0.14),
    ('Dishwasher', 'Appliance', 1.2, 1.5, 20, 0.10),
    ('Washing Machine', 'Appliance', 0.5, 1, 11, 0.14),
    ('Dryer', 'Appliance', 5.0, 1, 21, 0.10),
    ('Water Heater (Heat Pump)', 'Appliance', 0.5, 2, 11, 0.14),
    ('Refrigerator', 'Appliance', 0.15, 24, 0, 0.14),
    ('Lighting (LED)', 'Lighting', 0.3, 5, 18, 0.20),
]

usage_records = []
base_date = datetime.now() - timedelta(days=30)

for day_offset in range(30):
    day = base_date + timedelta(days=day_offset)
    is_weekend = day.weekday() >= 5

    for device_name, category, power_kw, daily_hours, start_hour, cost_rate in USAGE_PATTERNS:
        # Skip EV and pool pump some weekdays realistically
        if device_name == 'EV Charger (Level 2)' and random.random() < 0.2:
            continue
        if device_name == 'Pool Pump' and random.random() < 0.1:
            continue

        hours = daily_hours * random.uniform(0.85, 1.15)
        consumption = round(power_kw * hours, 3)
        cost = round(consumption * cost_rate, 4)
        ts = day.replace(hour=start_hour, minute=random.randint(0, 30), second=0)

        usage_records.append(EnergyUsage(
            timestamp=ts,
            device_name=device_name,
            device_category=category,
            consumption_kwh=consumption,
            duration_hours=round(hours, 2),
            cost_usd=cost,
        ))

with Session(engine) as session:
    session.add_all(usage_records)
    session.commit()

print(f'✅ Inserted {len(usage_records)} energy usage records.')

## 5. Verify the Data

In [ ]:
import pandas as pd

with Session(engine) as session:
    usage_df = pd.read_sql('SELECT device_category, SUM(consumption_kwh) as total_kwh, SUM(cost_usd) as total_cost FROM energy_usage GROUP BY device_category', engine)
    solar_df = pd.read_sql('SELECT DATE(timestamp) as date, SUM(generation_kwh) as daily_kwh FROM solar_generation GROUP BY DATE(timestamp) ORDER BY date DESC LIMIT 7', engine)
    price_df = pd.read_sql('SELECT tier, AVG(price_per_kwh) as avg_price FROM electricity_price GROUP BY tier', engine)

print('=== Energy Usage by Category (30 days) ===')
print(usage_df.to_string(index=False))
print()
print('=== Daily Solar Generation (Last 7 Days) ===')
print(solar_df.to_string(index=False))
print()
print('=== Average Price by Tier ===')
print(price_df.to_string(index=False))